In [10]:
import numpy as np
import os
import pandas as pd
import pickle
project_dir = os.path.expanduser('../')
load_dir = project_dir + "/data/value_functions"


In [11]:
SA_mask = pd.read_pickle(project_dir + "data/behavior_policy/SA_mask_shifted.pkl")
if isinstance(SA_mask, pd.DataFrame):
    SA_mask = SA_mask.values
    
for fname in sorted(os.listdir(load_dir)):
    if fname.endswith(".npz"):
        var_name = fname.replace(".npz", "")
        globals()[var_name] = np.load(os.path.join(load_dir, fname))["data"]

In [12]:
γ = 1

nS = 750
nA = 25

nS_term = 2
S_survival = 750
S_death = 751

nS_total = nS + nS_term

### Generate Policy

In [13]:
'''
    All optimal options (including ties) are recorded as optimal actions
'''
def generate_policy_from_Q(Q, SA_mask, tol=1e-6):
    nS, _ = Q.shape
    optimal_actions = {}
    
    for s in range(nS):
        valid_mask = (SA_mask[s])&(~np.isnan(Q[s]))
        valid_actions = np.flatnonzero(valid_mask)
        
        if len(valid_actions) == 0:
            optimal_actions[s] = []
            continue
        
        q_valid = Q[s][valid_actions]
        q_max = np.nanmax(q_valid)
        
        optimal_idx = valid_actions[np.abs(q_valid - q_max) <= tol]
        optimal_actions[s] = optimal_idx.tolist()
        
    return optimal_actions

In [14]:
π_plus = generate_policy_from_Q(Q_plus, SA_mask)
π_minus = generate_policy_from_Q(Q_minus, SA_mask)
π_mixed = generate_policy_from_Q(Q_mixed, SA_mask)

In [15]:
π_plus_gamma_99 = generate_policy_from_Q(Q_plus_gamma_99, SA_mask)
π_minus_gamma_99 = generate_policy_from_Q(Q_minus_gamma_99, SA_mask)
π_mixed_gamma_99 = generate_policy_from_Q(Q_mixed_gamma_99, SA_mask)

In [16]:
# quick sanity check for SA_mask handling: for each π[s], check if all included actions are SA_mask[s] true
def check_masking(π, SA_mask):
    for s in range(nS):
        valid_actions = np.flatnonzero(SA_mask[s])
        for oa in π[s]:
            if oa not in valid_actions:
                print(f"Warning: masking error at [{s}]")
    print("Passed")

check_masking(π_plus, SA_mask)
check_masking(π_minus, SA_mask)
check_masking(π_mixed, SA_mask)
check_masking(π_plus_gamma_99, SA_mask)
check_masking(π_minus_gamma_99, SA_mask)
check_masking(π_mixed_gamma_99, SA_mask)

Passed
Passed
Passed
Passed
Passed
Passed


In [ ]:
save_dir = os.path.join(project_dir, "data", "policies")
os.makedirs(save_dir, exist_ok=True)

policies = {
    "π_plus": π_plus,
    "π_minus": π_minus,
    "π_mixed": π_mixed,
    "π_plus_gamma_99": π_plus_gamma_99,
    "π_minus_gamma_99": π_minus_gamma_99,
    "π_mixed_gamma_99": π_mixed_gamma_99,
}

for name, policy in policies.items():
    with open(os.path.join(save_dir, f"{name}.pkl"), "wb") as f:
        pickle.dump(policy, f)

✅ Saved π_plus to ../data/policies/π_plus.pkl
✅ Saved π_minus to ../data/policies/π_minus.pkl
✅ Saved π_mixed to ../data/policies/π_mixed.pkl
✅ Saved π_plus_gamma_99 to ../data/policies/π_plus_gamma_99.pkl
✅ Saved π_minus_gamma_99 to ../data/policies/π_minus_gamma_99.pkl
✅ Saved π_mixed_gamma_99 to ../data/policies/π_mixed_gamma_99.pkl
